# 03 — ポーズ選別 / Pose Selection

`02_clustering.ipynb` で生成した `all_poses.csv` と `all_poses_cluster.sdf` を読み込み、
`project_config.toml` で定義したコンポーザブルフィルターを適用して、
選別したポーズを SDF + CSV で出力します。

Loads `all_poses.csv` and `all_poses_cluster.sdf` produced by `02_clustering.ipynb`,
applies composable filters defined in `project_config.toml`, and exports the
selected poses as SDF + CSV.

**以下の CONFIG セルのみ編集してください。 / Edit only the CONFIG cell below.**

In [ ]:
# CONFIG -----------------------------------------------------------------------
CONFIG_PATH = "../notebooks/templates/project_config.toml"  # ← edit this path
# ------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import PandasTools

from docking_analysis import (
    AnalysisConfig,
    ClusterRepresentativeFilter,
    InteractionFilter,
    ScoreFilter,
    apply_filters,
)

from docking_analysis.selection.filters import StrainEnergyFilter
from docking_analysis.analysis.strain import recommend_strain_thresholds, compute_strain_energy_h_relaxed
from docking_analysis.geometry.pose_geometry import add_geometry_to_df, classify_pose_geometry, plot_geometry_metrics
from docking_analysis.analysis.interaction_scoring import add_interaction_score_to_df
from docking_analysis.analysis.validation import compute_artifact_score
from docking_analysis.selection.diversity import greedy_select_with_quotas
from docking_analysis.selection.workflow import cluster_filtered_poses, plot_filtered_cluster_summary

In [ ]:
config = AnalysisConfig.from_toml(CONFIG_PATH)
print(f"Project           : {config.project_name}")
print(f"Interaction groups: {[g.label for g in config.interaction_groups]}")

## データの読み込み / Load data

In [ ]:
all_df = pd.read_csv(config.results_dir / "all_poses.csv")
mol_df = PandasTools.LoadSDF(str(config.results_dir / "all_poses_cluster.sdf"))

# Align mols list to match DataFrame row order
mol_name_to_mol = {m.GetProp("mol_name"): m for m in mol_df["ROMol"] if m is not None}
all_mols = [mol_name_to_mol.get(n) for n in all_df["mol_name"]]

print(f"Loaded {len(all_df)} poses, {len(mol_name_to_mol)} molecules")

## クラスター代表ポーズ（クラスターごとの最良スコア）
## Cluster representatives (best score per cluster)

In [ ]:
rep_df, rep_mols = apply_filters(
    all_df, all_mols,
    [ClusterRepresentativeFilter()]
)
print(f"Cluster representatives: {len(rep_df)}")
rep_df[["mol_name", "docking_score", "cluster_id"]].sort_values("docking_score").head(10)

In [ ]:
# Save cluster representatives
rep_df.to_csv(config.results_dir / "top_in_cluster.csv", index=False)

writer = Chem.SDWriter(str(config.results_dir / "top_in_cluster_mols.sdf"))
for mol in rep_mols:
    if mol is not None:
        writer.write(mol)
writer.close()
print("Saved: top_in_cluster.csv / top_in_cluster_mols.sdf")

## インタラクションに基づくフィルタリング / Interaction-based filtering

設定ファイルで定義されたインタラクショングループごとに出力セットを生成します。

Generates one output set per interaction group defined in the config.

In [ ]:
for group in config.interaction_groups:
    filt = InteractionFilter(residues=group.residues, require_all=group.require_all)
    sel_df, sel_mols = apply_filters(all_df, all_mols, [filt])

    print(f"\n[{group.label}] {len(sel_df)} poses")
    print(f"  residues     : {group.residues}")
    print(f"  require_all  : {group.require_all}")
    print(f"  score range  : {sel_df['docking_score'].min():.3f} – {sel_df['docking_score'].max():.3f}")

    label = group.label
    sel_df.to_csv(config.results_dir / f"selected_{label}.csv", index=False)

    writer = Chem.SDWriter(str(config.results_dir / f"selected_{label}_mols.sdf"))
    for mol in sel_mols:
        if mol is not None:
            writer.write(mol)
    writer.close()
    print(f"  Saved: selected_{label}.csv / selected_{label}_mols.sdf")

## クラスターごとのスコア分布（代表ポーズ）
## Score distribution per cluster (representatives)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(
    rep_df["cluster_id"], rep_df["docking_score"],
    alpha=0.7, s=50, color="steelblue",
)
ax.axhline(rep_df["docking_score"].mean(), color="red", linestyle="--", linewidth=1,
           label=f"mean = {rep_df['docking_score'].mean():.2f}")
ax.set_xlabel("Cluster ID")
ax.set_ylabel("Docking score (kcal/mol)")
ax.set_title("Top-scoring pose per cluster")
ax.legend()
plt.tight_layout()
plt.show()

## ひずみエネルギーフィルタ（H 固定プロトコル） / Strain Energy Filter (H-relaxed protocol)

重原子を固定したまま水素のみを緩和するプロトコルで、大きく柔軟な分子に適したひずみエネルギーを計算してフィルタリングします。

Filters poses by strain energy computed under the H-relaxed protocol (heavy atoms fixed, H free).

In [ ]:
# ひずみエネルギーをまだ計算していない場合は計算する
# Compute h-relaxed strain energy if not already present
if "strain_h_relaxed" not in all_df.columns:
    from rdkit.Chem import AllChem
    print("Computing H-relaxed strain energies (this may take a while)...")
    energies = []
    for mol in all_mols:
        if mol is None:
            energies.append(float("nan"))
            continue
        mol_h = AllChem.AddHs(mol, addCoords=True)
        e = compute_strain_energy_h_relaxed(mol_h)
        energies.append(e if e is not None else float("nan"))
    all_df["strain_h_relaxed"] = energies
    print("Done.")

# 分子量・回転可能結合数に応じた推奨閾値を確認
# Inspect recommended thresholds for a representative molecule
sample_mol = next((m for m in all_mols if m is not None), None)
if sample_mol is not None:
    thresholds = recommend_strain_thresholds(sample_mol)
    print(f"Recommended strain thresholds: warn={thresholds.warn_kcal} kcal/mol, reject={thresholds.reject_kcal} kcal/mol")

strain_filtered_df, strain_filtered_mols = apply_filters(
    all_df, all_mols,
    [StrainEnergyFilter(threshold=20.0, protocol="h_relaxed")],
)
print(f"Strain filter (H-relaxed): {len(all_df)} → {len(strain_filtered_df)} poses")

## ポーズ幾何学解析 / Pose Geometry Analysis

重心距離・埋没率・主軸角度を計算し、ポーズの空間配置を ACCEPT / REVIEW / REJECT に分類します。

Computes centroid distance, burial metrics, and principal axis angle to classify poses.

In [ ]:
# 参照重心（例：既知リガンドの重心）
# Reference centroid — use a known ligand or the box centre
import numpy as np
from docking_analysis.config.schema import GridBoxConfig

ref_centroid = None
if config.gridbox is not None:
    ref_centroid = np.array(config.gridbox.center)
    print(f"Using grid box centre as reference: {ref_centroid}")

if ref_centroid is not None:
    geom_df = add_geometry_to_df(
        rep_df,
        rep_mols,
        protein_pdb_paths={run.label: run.receptor for run in config.docking_runs},
        ref_centroid=ref_centroid,
    )
    
    fig = plot_geometry_metrics(
        geom_df,
        output_path=config.results_dir / "geometry_metrics.png",
    )
    plt.show()
    
    verdict_counts = geom_df["pose_verdict"].value_counts()
    print(verdict_counts)
    
    # ACCEPT のみを以降の解析に使用
    # Keep only ACCEPT poses for downstream analysis
    accepted_geom_df = geom_df[geom_df["pose_verdict"] == "ACCEPT"].copy()
    print(f"Geometry filter: {len(rep_df)} → {len(accepted_geom_df)} poses (ACCEPT)")
else:
    accepted_geom_df = rep_df.copy()
    print("No reference centroid available — skipping geometry filter.")

## 残基重み付きインタラクションスコア / Residue-Weighted Interaction Score

必須残基（×3）と推奨残基（×1）の ProLIF 接触フラグを加重合計したインタラクションスコアを計算します。

Computes a weighted interaction score from required (×3) and recommended (×1) ProLIF contact flags.

In [ ]:
# 設定ファイルの最初のインタラクショングループから必須・推奨残基を取得
# Get required/recommended residues from the first interaction group in config
if config.interaction_groups:
    first_group = config.interaction_groups[0]
    required_cols  = [c for c in all_df.columns if any(r in c for r in first_group.residues)]
    recommended_cols = []  # 推奨残基があればここに追加 / add recommended residue cols here
else:
    # 手動指定 / manual specification
    required_cols    = []  # 例: ["GLN30_HBAcceptor", "ARG38_HBDonor"]
    recommended_cols = []

if required_cols:
    scored_df = add_interaction_score_to_df(
        rep_df,
        required_cols=required_cols,
        recommended_cols=recommended_cols,
        output_col="interaction_score",
    )
    print(scored_df[["mol_name", "docking_score", "interaction_score"]].sort_values("interaction_score", ascending=False).head(10))
else:
    scored_df = rep_df.copy()
    scored_df["interaction_score"] = 0.0
    print("No required residue columns found — interaction_score set to 0.")

## アーティファクトスコア / Artifact Score

重心距離・ひずみエネルギー・インタラクションスコアを 0–1 正規化して統合した「低いほど良い」スコアを計算します。

Integrates centroid distance, strain energy and interaction score into a single low-is-better artifact score.

In [ ]:
artifact_scores = compute_artifact_score(
    scored_df,
    centroid_col="centroid_distance" if "centroid_distance" in scored_df.columns else None,
    strain_col="strain_h_relaxed"    if "strain_h_relaxed"    in scored_df.columns else None,
    interaction_col="interaction_score",
)
scored_df["artifact_score"] = artifact_scores

print(scored_df[["mol_name", "docking_score", "interaction_score", "artifact_score"]]
      .sort_values("artifact_score").head(10))

## クォータ付き多様性選択 / Diversity Selection with Per-Receptor Quotas

受容体（またはインタラクショングループ）ごとに min/max クォータを守りながら Tanimoto 多様性フィルタ付きで代表化合物を選択します。

Selects diverse representatives per receptor/group while respecting per-group min/max quotas.

In [ ]:
# 各グループから最低 1 件・最大 5 件を選択する例
# Example: select min=1, max=5 per receptor group
if "receptor" in scored_df.columns and "SMILES" in scored_df.columns:
    diverse_df = greedy_select_with_quotas(
        scored_df,
        score_col="artifact_score",
        group_col="receptor",
        quotas={"default": (1, 5)},  # (min, max) per group
        similarity_threshold=0.4,    # Tanimoto 類似度フィルタ
        smiles_col="SMILES",
        ascending=True,              # artifact_score は低いほど良い
    )
    print(f"Diversity selection: {len(scored_df)} → {len(diverse_df)} poses")
    diverse_df[["mol_name", "receptor", "docking_score", "artifact_score"]].head(10)
else:
    print("Columns 'receptor' and 'SMILES' required — skipping quota diversity selection.")

## フィルタ優先再クラスタリング / Filter-First Reclustering

ひずみ・幾何学フィルタを通過したポーズのみを対象に再クラスタリングし、各クラスタの代表ポーズを選択します。

Re-clusters only the filtered poses and selects top representatives per cluster.

In [ ]:
def quality_filter(df, mols):
    """ひずみ・幾何学が通過したポーズのみを残す / Keep only quality-passed poses."""
    mask = pd.Series(True, index=df.index)
    if "strain_h_relaxed" in df.columns:
        mask &= df["strain_h_relaxed"].fillna(999) < 20.0
    if "pose_verdict" in df.columns:
        mask &= df["pose_verdict"].isin(["ACCEPT", "REVIEW"])
    idx = df[mask].index
    return df.loc[idx].reset_index(drop=True), [mols[i] for i in idx]

try:
    selected_df, cluster_labels = cluster_filtered_poses(
        all_df, all_mols,
        filter_fn=quality_filter,
        rmsd_threshold=optimal_threshold,
        score_col="docking_score",
        select_top_n=1,
    )
    print(f"Filter-first reclustering: {len(all_df)} → {len(cluster_labels)} → {len(selected_df)} poses")
    
    # サマリー図 / Summary figure
    if "pose_verdict" in selected_df.columns:
        fig = plot_filtered_cluster_summary(
            selected_df,
            metrics=["centroid_distance", "strain_h_relaxed", "principal_axis_angle", "docking_score"],
            verdict_col="pose_verdict",
            output_path=config.results_dir / "filtered_cluster_summary.png",
        )
        plt.show()
except ValueError as e:
    print(f"Filter-first reclustering skipped: {e}")